In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import numpy as np
import numpy as np
from sklearn.metrics import accuracy_score

In [3]:
df = pd.read_csv('cleaned_twitter_data_bert.csv')

In [4]:
df.head()

,target,text,cleaned_text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",- A that's a bummer. You shoulda got David Car...
1,0,is upset that he can't update his Facebook by ...,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...,I dived many times for the ball. Managed to sa...
3,0,my whole body feels itchy and like its on fire,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all....","no, it's not behaving at all. i'm mad. why am ..."


In [5]:
df.shape

(1600000, 3)

In [6]:
texts = df['cleaned_text'].astype(str).tolist()
labels = df['target'].tolist()

In [7]:
len(texts)

1600000

In [8]:
len(labels)

1600000

In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)


In [10]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

In [11]:
class TwitterDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = str(self.texts[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

        return item

    def __len__(self):
        return len(self.labels)

In [12]:
train_dataset = TwitterDataset(train_texts, train_labels, tokenizer)
test_dataset = TwitterDataset(test_texts, test_labels, tokenizer)

In [13]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
# Metric
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1
    }

In [17]:
training_args = TrainingArguments(
    output_dir="./bert_results_2",
    num_train_epochs=10,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",          # save after every epoch
    save_total_limit=2,             # keep only latest 2 checkpoints
    logging_steps=500,
    fp16=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    load_best_model_at_end=True,
    report_to="none",
    learning_rate=2e-5,
    weight_decay=0.01
)

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [19]:
trainer.train()
model.save_pretrained("./final_DistilBERT_model")
tokenizer.save_pretrained("./final_DistilBERT_model")

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.325894,0.315298,0.863684,0.863379
2,0.289770,0.310754,0.867581,0.866190
3,0.258214,0.321208,0.867572,0.866749
4,0.225595,0.339541,0.866028,0.865398


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./final_DistilBERT_model/tokenizer_config.json',
 './final_DistilBERT_model/tokenizer.json')

In [20]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.3107493817806244, 'eval_accuracy': 0.867596875, 'eval_f1': 0.8662075237544881, 'eval_runtime': 151.2704, 'eval_samples_per_second': 2115.417, 'eval_steps_per_second': 66.107, 'epoch': 4.0}


In [22]:
# check gpu available
import torch
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(device)

cuda
